# Message Queues for ML Pipelines

## Why Message Queues in ML?
ML inference can be slow (GPU-heavy models), variable in load, and needs to be decoupled from the request cycle. Message queues enable:
- **Async inference**: Return immediately, process later
- **Load leveling**: Absorb traffic spikes
- **Retry logic**: Reprocess failed inference jobs
- **Fan-out**: Send predictions to multiple consumers

## Tools
| Tool | Type | Best For |
|------|------|----------|
| **Apache Kafka** | Distributed log | High-throughput streaming, event sourcing |
| **RabbitMQ** | AMQP broker | Task queues, work distribution |
| **Celery** | Task queue | Python async tasks with Redis/RabbitMQ |
| **Redis Streams** | Lightweight streaming | Simple streaming, caching + queuing |
| **AWS SQS** | Managed queue | Serverless, managed |
| **Google Pub/Sub** | Managed pub/sub | GCP ecosystem |

## Apache Kafka Architecture

```
Producer → [Topic: inference-requests]
             Partition 0: [msg1, msg2, msg3]
             Partition 1: [msg4, msg5, msg6]
                    ↓
          Consumer Group: ml-workers
             Worker 1 ← Partition 0
             Worker 2 ← Partition 1
                    ↓
          [Topic: inference-results]
                    ↓
          Consumer Group: downstream-services
```

### Key Concepts
- **Topic**: Named stream of records
- **Partition**: Ordered, immutable log within a topic (enables parallelism)
- **Offset**: Position of a message in a partition
- **Consumer Group**: Multiple consumers sharing partition load
- **Replication Factor**: Fault tolerance copies

In [1]:
# Kafka Producer for ML inference requests
# pip install kafka-python

KAFKA_PRODUCER = '''
from kafka import KafkaProducer
import json
import numpy as np
import time

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    acks='all',            # Wait for all replicas
    retries=3,
    max_in_flight_requests_per_connection=1  # Preserve ordering
)

def send_inference_request(request_id: str, features: list, model_version: str = 'v1'):
    message = {
        'request_id': request_id,
        'features': features,
        'model_version': model_version,
        'timestamp': time.time()
    }
    
    future = producer.send(
        topic='inference-requests',
        key=request_id,
        value=message
    )
    record_metadata = future.get(timeout=10)
    print(f"Sent to {record_metadata.topic}:{record_metadata.partition}@{record_metadata.offset}")
    return record_metadata

# Send batch of requests
for i in range(10):
    features = np.random.rand(4).tolist()
    send_inference_request(f"req_{i:04d}", features)

producer.flush()
producer.close()
'''

print('Kafka producer code:')
print(KAFKA_PRODUCER[:400] + '...')

Kafka producer code:

from kafka import KafkaProducer
import json
import numpy as np
import time

producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    acks='all',            # Wait for all replicas
    retries=3,
    max_in_flight_requests_per_connection=1  # Preserve ordering...


In [2]:
# Kafka Consumer (ML Worker)
KAFKA_CONSUMER = '''
from kafka import KafkaConsumer, KafkaProducer
import json
import numpy as np
import joblib
import time

# Load model once
model = joblib.load('/models/iris_classifier.joblib')

consumer = KafkaConsumer(
    'inference-requests',
    bootstrap_servers=['localhost:9092'],
    group_id='ml-workers',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest',
    enable_auto_commit=False,  # Manual commit for reliability
    max_poll_records=50
)

result_producer = KafkaProducer(
    bootstrap_servers=['localhost:9092'],
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print('ML Worker started, waiting for inference requests...')

for message in consumer:
    try:
        data = message.value
        request_id = data['request_id']
        features = data['features']
        
        # Run inference
        start = time.time()
        X = np.array([features])
        prediction = model.predict(X)[0]
        probability = model.predict_proba(X)[0].tolist()
        latency_ms = (time.time() - start) * 1000
        
        # Send result
        result = {
            'request_id': request_id,
            'prediction': int(prediction),
            'probability': probability,
            'latency_ms': latency_ms,
            'model_version': data.get('model_version', 'unknown')
        }
        result_producer.send('inference-results', value=result, key=request_id.encode())
        
        # Commit offset manually
        consumer.commit()
        print(f"Processed {request_id}: pred={prediction}, latency={latency_ms:.1f}ms")
    
    except Exception as e:
        print(f"Error processing {message}: {e}")
        # Dead letter queue
        result_producer.send('inference-dead-letter', value={'error': str(e), 'original': message.value})
'''

print('Kafka consumer/ML worker code defined')

Kafka consumer/ML worker code defined


## Celery for Async ML Tasks

Celery is the most popular Python task queue, using Redis or RabbitMQ as the broker.

```python
# celery_app.py
from celery import Celery
import numpy as np
import joblib

app = Celery(
    'ml_tasks',
    broker='redis://localhost:6379/0',    # Task queue
    backend='redis://localhost:6379/1',   # Result storage
)

app.conf.update(
    task_serializer='json',
    result_serializer='json',
    task_routes={'ml_tasks.predict': {'queue': 'gpu-queue'}},
    task_time_limit=60,
    task_soft_time_limit=50,
    worker_prefetch_multiplier=1,  # One task at a time for GPU tasks
)

# Load model once per worker process
MODEL = None
def get_model():
    global MODEL
    if MODEL is None:
        MODEL = joblib.load('/models/iris_classifier.joblib')
    return MODEL

@app.task(bind=True, max_retries=3, name='ml_tasks.predict')
def predict(self, features: list, model_version: str = 'v1'):
    try:
        model = get_model()
        X = np.array([features])
        prediction = model.predict(X)[0]
        proba = model.predict_proba(X)[0].tolist()
        return {'prediction': int(prediction), 'probabilities': proba}
    except Exception as exc:
        raise self.retry(exc=exc, countdown=2 ** self.request.retries)

@app.task
def batch_predict(feature_list: list):
    model = get_model()
    X = np.array(feature_list)
    predictions = model.predict(X).tolist()
    return {'predictions': predictions}
```

```python
# Calling Celery tasks from FastAPI
from fastapi import FastAPI, BackgroundTasks
from celery_app import predict, batch_predict

app = FastAPI()

@app.post('/predict')
async def async_predict(features: list):
    # Dispatch task asynchronously
    task = predict.delay(features)
    return {'task_id': task.id, 'status': 'queued'}

@app.get('/result/{task_id}')
async def get_result(task_id: str):
    from celery.result import AsyncResult
    result = AsyncResult(task_id)
    if result.ready():
        return {'status': 'done', 'result': result.get()}
    return {'status': result.state}
```

```bash
# Start workers
celery -A celery_app worker --loglevel=info --concurrency=4 -Q gpu-queue

# Celery Beat (scheduled tasks)
celery -A celery_app beat --loglevel=info

# Monitor with Flower UI
pip install flower
celery -A celery_app flower --port=5555
```

## Redis Streams for Lightweight Streaming

```python
import redis
import json
import time

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# Producer: Add inference request to stream
def produce_request(stream_name: str, request_id: str, features: list):
    r.xadd(
        stream_name,
        fields={
            'request_id': request_id,
            'features': json.dumps(features),
            'timestamp': str(time.time())
        },
        maxlen=10000  # Keep only last 10k messages
    )

# Consumer: Read from stream
def consume_requests(stream_name: str, consumer_group: str, consumer_name: str):
    # Create consumer group (once)
    try:
        r.xgroup_create(stream_name, consumer_group, id='0', mkstream=True)
    except redis.exceptions.ResponseError:
        pass  # Group already exists
    
    while True:
        messages = r.xreadgroup(
            consumer_group, consumer_name,
            {stream_name: '>'},  # '>' = only new, undelivered messages
            count=10,
            block=5000  # Wait 5s for new messages
        )
        for stream, msgs in (messages or []):
            for msg_id, fields in msgs:
                # Process
                features = json.loads(fields['features'])
                print(f"Processing: {fields['request_id']}, features: {features[:2]}")
                
                # Acknowledge after processing
                r.xack(stream_name, consumer_group, msg_id)
```

## Additional Learning Resources

### Kafka
- [Kafka Docs](https://kafka.apache.org/documentation/)
- [Confluent Kafka Python](https://docs.confluent.io/kafka-clients/python/current/overview.html)
- [Kafka: The Definitive Guide](https://www.oreilly.com/library/view/kafka-the-definitive/9781492043072/) Free PDF from Confluent

### Celery
- [Celery Docs](https://docs.celeryq.dev/)
- [Celery Best Practices](https://denibertovic.com/posts/celery-best-practices/)

### Redis
- [Redis Streams Docs](https://redis.io/docs/data-types/streams/)
- [Redis for ML Blog](https://redis.io/blog/use-redis-as-a-feature-store/)

### Books
- [Designing Data-Intensive Applications Martin Kleppmann](https://www.oreilly.com/library/view/designing-data-intensive-applications/9781491903063/) Chapter 11 on streaming